# Shoulder Flexion/Extension — Recording Type Comparison (CSV)
Reads a CSV with columns: `Laterality · Recording · Movement · Side · Rep · Peak_deg · Valley_deg · ROM_deg`

Generates three figures (ROM · Maximum · Minimum), each with Left and Right subplots
showing 4 raincloud plots per side:
**Continuous Unilateral · Continuous Bilateral · Individual Unilateral · Individual Bilateral**

## 1. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from scipy.stats import gaussian_kde

## 2. Configuration
**Edit only this cell.**

In [ ]:
CSV_PATH   = r"C:\Projects\data_organized\grab 3_c3d\Excels\Shoulder_Flex_Ext.csv"   # path to your CSV file
OUTPUT_DIR = None   # set to r"C:\path\to\output" to save PNGs; None = show only
DPI        = 150

# Values in the CSV are stored as centi-degrees (×100). Set to False if already in degrees.
VALUES_ARE_CENTIDEGREES = True

## 3. Parse CSV

In [ ]:
# ── Read ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8-sig")

# Keep only the columns we need and drop fully-empty rows
df = df[["Laterality", "Recording", "Side", "Peak_deg", "Valley_deg", "ROM_deg"]].dropna(subset=["Side"])

# Convert to numeric (coerce errors → NaN)
for col in ("Peak_deg", "Valley_deg", "ROM_deg"):
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Divide by 100 if values are stored as centi-degrees
if VALUES_ARE_CENTIDEGREES:
    df[["Peak_deg", "Valley_deg", "ROM_deg"]] /= 100.0

print(f"Rows loaded: {len(df)}")
print(df.groupby(["Laterality", "Recording", "Side"]).size().to_string())

## 4. Build recording_data dict

In [ ]:
RECORDING_TYPES = [
    "Continuous Unilateral",
    "Continuous Bilateral",
    "Individual Unilateral",
    "Individual Bilateral",
]
SIDES = ["Left", "Right"]

# Map CSV column values → recording type label
_REC_TYPE_MAP = {
    ("Unilateral", "Continuous"):  "Continuous Unilateral",
    ("Bilateral",  "Continuous"):  "Continuous Bilateral",
    ("Unilateral", "Individual"): "Individual Unilateral",
    ("Bilateral",  "Individual"): "Individual Bilateral",
}


def _agg(values: list[float]) -> dict:
    v = np.array([x for x in values if x is not None and not np.isnan(x)])
    return {
        "values": values,
        "mean": float(np.mean(v))        if v.size > 0 else float("nan"),
        "sd":   float(np.std(v, ddof=1)) if v.size > 1 else 0.0,
        "min":  float(np.min(v))          if v.size > 0 else float("nan"),
        "max":  float(np.max(v))          if v.size > 0 else float("nan"),
    }


def _empty_result() -> dict:
    empty = {"values": [], "mean": float("nan"), "sd": 0.0,
             "min": float("nan"), "max": float("nan")}
    return {"extended": {"rom": {**empty}, "peak": {**empty}, "valley": {**empty}}}


recording_data: dict[tuple[str, str], dict] = {}

for side in SIDES:
    for rec_type in RECORDING_TYPES:
        key = (side, rec_type)
        # Find matching (Laterality, Recording) pair
        csv_key = next(
            (k for k, v in _REC_TYPE_MAP.items() if v == rec_type), None
        )
        if csv_key is None:
            recording_data[key] = _empty_result()
            continue

        lat, rec = csv_key
        mask = (df["Laterality"] == lat) & (df["Recording"] == rec) & (df["Side"] == side)
        sub = df[mask]

        recording_data[key] = {
            "extended": {
                "rom":    _agg(sub["ROM_deg"].tolist()),
                "peak":   _agg(sub["Peak_deg"].tolist()),
                "valley": _agg(sub["Valley_deg"].tolist()),
            }
        }

# ── Summary table ──────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print(f"{'Side':<8} {'Recording Type':<26} {'Metric':<8} {'n':>4} {'Mean':>8} {'SD':>8}")
print("=" * 72)
for side in SIDES:
    for rec_type in RECORDING_TYPES:
        result = recording_data[(side, rec_type)]
        for metric in ("rom", "peak", "valley"):
            m = result["extended"][metric]
            valid = [v for v in m["values"] if v is not None and not np.isnan(float(v))]
            n        = len(valid)
            mean_str = f"{m['mean']:.1f}°" if not np.isnan(m["mean"]) else "—"
            sd_str   = f"{m['sd']:.1f}°"   if n > 1 else "—"
            print(f"{side:<8} {rec_type:<26} {metric:<8} {n:>4} {mean_str:>8} {sd_str:>8}")

## 5. Color palette

In [ ]:
# (fill_color, dark_outline_color) per side × recording type
COLORS: dict[str, dict[str, tuple[str, str]]] = {
    "Left": {
        "Continuous Unilateral":  ("#922B21", "#641E16"),
        "Continuous Bilateral":   ("#C0392B", "#922B21"),
        "Individual Unilateral":  ("#E74C3C", "#C0392B"),
        "Individual Bilateral":   ("#F1948A", "#E74C3C"),
    },
    "Right": {
        "Continuous Unilateral":  ("#145A32", "#0B3B21"),
        "Continuous Bilateral":   ("#1E8449", "#145A32"),
        "Individual Unilateral":  ("#2ECC71", "#1E8449"),
        "Individual Bilateral":   ("#82E0AA", "#2ECC71"),
    },
}

SIDE_LABEL_COLORS = {"Left": "#641E16", "Right": "#0B3B21"}

## 6. Raincloud drawing function

In [ ]:
_V_AMP  = 0.25
_BW     = 0.025
_SC_OFF = 0.08
_SC_JIT = 0.025


def _draw_raincloud(
    ax: plt.Axes,
    vals: np.ndarray,
    x_c: float,
    color: str,
    color_dark: str,
    rng: np.random.Generator,
) -> None:
    """Violin (left) + boxplot (centre) + jittered scatter (right)."""
    vals = np.asarray(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if vals.size == 0:
        return

    # 1. Half-violin to the left of x_c
    if vals.size >= 2:
        kde    = gaussian_kde(vals, bw_method="scott")
        spread = max(float(vals.std()) * 0.5, 0.5)
        y_grid = np.linspace(vals.min() - spread, vals.max() + spread, 300)
        dens   = kde(y_grid)
        peak_d = dens.max()
        if peak_d > 0:
            kde_x = x_c - (dens / peak_d) * _V_AMP
            ax.fill_betweenx(y_grid, kde_x, x_c, color=color, alpha=0.35, zorder=2)
            ax.plot(kde_x, y_grid, color=color, alpha=0.70, linewidth=0.9, zorder=3)

    # 2. Vertical boxplot
    q1, q2, q3 = np.percentile(vals, [25, 50, 75])
    iqr      = q3 - q1
    fence_lo = q1 - 1.5 * iqr
    fence_hi = q3 + 1.5 * iqr
    w_lo = vals[vals >= fence_lo].min() if np.any(vals >= fence_lo) else q1
    w_hi = vals[vals <= fence_hi].max() if np.any(vals <= fence_hi) else q3

    ax.plot([x_c, x_c], [w_lo, q1], color=color, lw=1.2, zorder=4)
    ax.plot([x_c, x_c], [q3, w_hi], color=color, lw=1.2, zorder=4)
    for wy in (w_lo, w_hi):
        ax.plot([x_c - _BW, x_c + _BW], [wy, wy], color=color, lw=1.2, zorder=4)
    ax.add_patch(Rectangle(
        (x_c - _BW, q1), 2 * _BW, q3 - q1,
        facecolor="white", edgecolor=color, linewidth=1.5, zorder=5,
    ))
    ax.plot([x_c - _BW, x_c + _BW], [q2, q2], color=color, lw=2.0, zorder=6)

    mean_val = float(vals.mean())
    ax.text(x_c, w_hi, f"{mean_val:.1f}°",
            ha="center", va="bottom", fontsize=7.5,
            fontweight="bold", color=color_dark, zorder=7)

    outliers = vals[(vals < w_lo) | (vals > w_hi)]
    if outliers.size:
        ax.scatter(np.full(outliers.size, x_c), outliers,
                   color=color, s=18, marker="D", alpha=0.85, zorder=7)

    # 3. Jittered scatter to the right
    jitter = rng.uniform(-_SC_JIT, _SC_JIT, size=vals.size)
    ax.scatter(x_c + _SC_OFF + jitter, vals,
               color=color, s=25, alpha=0.80, zorder=3,
               edgecolors=color_dark, linewidths=0.4)

## 7. Plot function

In [ ]:
_SPACING = 0.55
_n = len(RECORDING_TYPES)
_SUBPLOT_X: dict[str, float] = {rt: i * _SPACING for i, rt in enumerate(RECORDING_TYPES)}

_subplot_x_min = -_V_AMP - 0.18
_subplot_x_max = (_n - 1) * _SPACING + _SC_OFF + _SC_JIT + 0.18

_XTICK_LABELS = {
    "Continuous Unilateral":  "Cont.\nUnil.",
    "Continuous Bilateral":   "Cont.\nBil.",
    "Individual Unilateral":  "Indiv.\nUnil.",
    "Individual Bilateral":   "Indiv.\nBil.",
}


def plot_metric(metric_key: str, metric_title: str) -> plt.Figure:
    """
    One figure with 2 subplots (Left | Right).
    Each subplot has 4 raincloud plots, one per recording type.
    """
    rng = np.random.default_rng(42)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        f"Shoulder Flexion/Extension — {metric_title}",
        fontsize=13, fontweight="bold",
    )
    fig.subplots_adjust(wspace=0.35, top=0.88, bottom=0.22)

    for ax, side in zip(axes, SIDES):
        ax.set_title(side, fontsize=11, fontweight="bold",
                     color=SIDE_LABEL_COLORS[side])

        tick_x: list[float] = []
        tick_labels: list[str] = []

        for rec_type in RECORDING_TYPES:
            raw = (recording_data[(side, rec_type)]["extended"]
                   .get(metric_key, {}).get("values", []))
            vals = np.array(
                [v for v in raw if v is not None and not np.isnan(float(v))],
                dtype=float,
            )
            x_c = _SUBPLOT_X[rec_type]
            color, color_dark = COLORS[side][rec_type]
            _draw_raincloud(ax, vals, x_c, color, color_dark, rng)
            tick_x.append(x_c)
            tick_labels.append(_XTICK_LABELS[rec_type])

        ax.set_xticks(tick_x)
        ax.set_xticklabels(tick_labels, fontsize=8)
        ax.set_xlim(_subplot_x_min, _subplot_x_max)
        ax.tick_params(axis="x", length=0)
        ax.margins(y=0.25)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
        ax.set_ylabel("Angle (°)", fontsize=9)

    legend_handles = [
        mpatches.Patch(facecolor=COLORS["Left"][rt][0], label=rt)
        for rt in RECORDING_TYPES
    ]
    fig.legend(
        handles=legend_handles,
        fontsize=9, ncol=4,
        loc="lower center", bbox_to_anchor=(0.5, -0.02),
        frameon=True, handlelength=1.2, handletextpad=0.5,
        columnspacing=1.2, borderpad=0.5,
    )
    return fig

## 8. Generate the three figures

In [ ]:
METRICS_TO_PLOT = [
    ("rom",    "Range of Motion (ROM)"),
    ("peak",   "Maximum Angle (Flexion)"),
    ("valley", "Minimum Angle (Extension)"),
]

if OUTPUT_DIR:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

for metric_key, metric_title in METRICS_TO_PLOT:
    print(f"\nGenerating: {metric_title}")
    fig = plot_metric(metric_key, metric_title)
    if OUTPUT_DIR:
        fname = Path(OUTPUT_DIR) / f"shoulder_flexext_{metric_key}.png"
        fig.savefig(fname, dpi=DPI, bbox_inches="tight")
        print(f"  Saved: {fname}")
    plt.show()